# 05 - Évaluation Critique et Généralisation

## Objectif
1. Évaluer le modèle sur la catastrophe réservée (Sewol)
2. Diagnostiquer le gap de performance (covariate shift, concept drift)
3. Analyser les biais (genre, classe, âge)
4. Aborder le paradoxe éthique "femmes et enfants d'abord"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
import sys, os
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))
from src.data_processing import *
from src.feature_engineering import *
from src.models import get_base_models
from src.evaluation import (
    comprehensive_evaluation, compute_disparate_impact,
    analyze_bias_comprehensive, diagnose_performance_gap,
    covariate_shift_analysis
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)

print('Modules chargés.')

In [ ]:
# Préparer les données comme dans le notebook 04
titanic_clean = clean_titanic(load_titanic())
lusitania_clean = clean_lusitania(load_lusitania())
sewol_clean = clean_sewol(load_sewol())
unified = create_unified_dataset(titanic_clean, lusitania_clean, sewol_clean)
df_enriched = full_feature_engineering_pipeline(unified)

# Split
train_mask = df_enriched['disaster'].isin(['titanic', 'lusitania'])
test_mask = df_enriched['disaster'] == 'sewol'
df_train = df_enriched[train_mask].copy()
df_test = df_enriched[test_mask].copy()

X_train, y_train, scaler, feature_names = prepare_features_for_modeling(df_train, scale=True)
X_test_raw, y_test, _, _ = prepare_features_for_modeling(df_test, scale=False)
X_test = pd.DataFrame(scaler.transform(X_test_raw), columns=feature_names, index=X_test_raw.index)

# Entraîner le meilleur modèle (XGBoost pooled)
from xgboost import XGBClassifier
best_model = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    use_label_encoder=False, eval_metric='logloss', verbosity=0
)
best_model.fit(X_train, y_train)

print(f"Train: {len(X_train)}, Test (Sewol holdout): {len(X_test)}")
print(f"Train accuracy: {best_model.score(X_train, y_train):.4f}")
print(f"Test accuracy: {best_model.score(X_test, y_test):.4f}")

---
## 1. Évaluation Complète sur la Catastrophe Réservée

In [ ]:
# Évaluation exhaustive
eval_results = comprehensive_evaluation(best_model, X_test, y_test, 'XGBoost')

print("=== Métriques sur Sewol (Catastrophe Réservée) ===")
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'avg_precision', 'brier_score']:
    if metric in eval_results:
        print(f"  {metric:20s}: {eval_results[metric]:.4f}")

print(f"\n=== Matrice de Confusion ===")
print(eval_results['confusion_matrix'])

In [ ]:
# Visualisations d'évaluation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Évaluation sur Sewol (Catastrophe Réservée)', fontsize=14, fontweight='bold')

# Matrice de confusion
ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test, ax=axes[0],
    display_labels=['Décédé', 'Survécu'], cmap='Blues'
)
axes[0].set_title('Matrice de Confusion')

# Courbe ROC
RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=axes[1])
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title(f'Courbe ROC (AUC={eval_results.get("roc_auc", 0):.3f})')

# Courbe Precision-Recall
PrecisionRecallDisplay.from_estimator(best_model, X_test, y_test, ax=axes[2])
axes[2].set_title(f'Courbe PR (AP={eval_results.get("avg_precision", 0):.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# Calibration plot
if 'calibration' in eval_results:
    prob_true, prob_pred = eval_results['calibration']

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(prob_pred, prob_true, 'o-', color='steelblue', label='XGBoost')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Calibration parfaite')
    ax.set_xlabel('Probabilité prédite')
    ax.set_ylabel('Proportion réelle de survie')
    ax.set_title('Courbe de Calibration sur le Sewol')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("Note : Un bon modèle calibré suit la diagonale.")
    print("Les écarts indiquent que le modèle est trop confiant ou pas assez.")

---
## 2. Diagnostic du Gap de Performance

### Est-ce surprenant que le modèle performe moins bien sur le Sewol ?

**Non, ce n'est pas surprenant** pour les raisons suivantes :

1. **Covariate shift** : Les distributions des features sont très différentes (lycéens de 17 ans vs passagers variés du Titanic)
2. **Concept drift** : La relation gender→survival a changé (pas de "women first" sur le Sewol)
3. **100 ans d'écart** : Les protocoles de sécurité, les types de navires et les contextes sociaux ont radicalement changé

In [ ]:
# Diagnostic complet
diagnostic = diagnose_performance_gap(
    XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                  random_state=42, use_label_encoder=False, eval_metric='logloss', verbosity=0),
    X_train, y_train, X_test, y_test,
    df_train, df_test, feature_names
)

print("=== Diagnostic du Gap ===")
print(f"CV Accuracy (Titanic+Lusitania): {diagnostic['cv_mean']:.4f} ± {diagnostic['cv_std']:.4f}")
print(f"Test Accuracy (Sewol holdout):   {diagnostic['test_score']:.4f}")
print(f"Gap:                              {diagnostic['gap']:.4f}")
print(f"\nBase rate train: {diagnostic['train_base_rate']:.4f}")
print(f"Base rate test:  {diagnostic['test_base_rate']:.4f}")

In [ ]:
# Analyse du covariate shift
shift_df = covariate_shift_analysis(X_train, X_test, feature_names)

print("=== Analyse du Covariate Shift (Test KS) ===")
print(shift_df[['feature', 'ks_statistic', 'p_value', 'significant_shift',
                'train_mean', 'test_mean']].to_string(index=False))

In [ ]:
# Visualisation du covariate shift
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Covariate Shift : Train (Titanic+Lusitania) vs Test (Sewol)', fontsize=14, fontweight='bold')

top_shifts = shift_df.head(6)['feature'].tolist()
for i, feat in enumerate(top_shifts):
    ax = axes[i // 3, i % 3]
    ax.hist(X_train[feat], bins=30, alpha=0.5, label='Train', color='steelblue', density=True)
    ax.hist(X_test[feat], bins=30, alpha=0.5, label='Test (Sewol)', color='coral', density=True)
    ks_val = shift_df[shift_df['feature'] == feat]['ks_statistic'].values[0]
    ax.set_title(f'{feat} (KS={ks_val:.3f})')
    ax.legend()

plt.tight_layout()
plt.show()

### Expériences de diagnostic

Pour déterminer si la baisse de performance est due à des distributions différentes, des frontières de décision différentes, des features manquantes, ou des problèmes de taille d'échantillon :

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Expérience 1 : Entraîner uniquement sur le Sewol (CV interne)
print("=== Expérience 1 : CV interne sur Sewol seul ===")
model_sewol = XGBClassifier(n_estimators=100, max_depth=4, random_state=42,
                            use_label_encoder=False, eval_metric='logloss', verbosity=0)
X_sewol_all, y_sewol_all, _, _ = prepare_features_for_modeling(df_test, scale=True)
sewol_cv = cross_val_score(model_sewol, X_sewol_all, y_sewol_all, cv=skf, scoring='accuracy')
print(f"Sewol CV: {sewol_cv.mean():.4f} ± {sewol_cv.std():.4f}")
print("→ Si ce score est élevé, le problème est le distribution shift, pas le manque de signal.")

# Expérience 2 : Réduire les features aux features communes universelles
print("\n=== Expérience 2 : Features universelles uniquement ===")
universal_features = ['age', 'sex_binary', 'social_class', 'is_crew', 'group_size', 'is_alone']
available_universal = [f for f in universal_features if f in feature_names]
model_universal = XGBClassifier(n_estimators=200, max_depth=6, random_state=42,
                                use_label_encoder=False, eval_metric='logloss', verbosity=0)
model_universal.fit(X_train[available_universal], y_train)
universal_score = model_universal.score(X_test[available_universal], y_test)
print(f"Score avec features universelles: {universal_score:.4f}")
print(f"Score avec toutes les features:   {diagnostic['test_score']:.4f}")
print("→ Si le score universelles est similaire, les features contextuelles n'aident pas pour le Sewol.")

# Expérience 3 : Sous-échantillonner l'entraînement pour tester l'effet de taille
print("\n=== Expérience 3 : Effet de la taille d'échantillon ===")
for frac in [0.1, 0.25, 0.5, 0.75, 1.0]:
    n_samples = int(len(X_train) * frac)
    idx = np.random.RandomState(42).choice(len(X_train), n_samples, replace=False)
    model_sub = XGBClassifier(n_estimators=200, max_depth=6, random_state=42,
                              use_label_encoder=False, eval_metric='logloss', verbosity=0)
    model_sub.fit(X_train.iloc[idx], y_train.iloc[idx])
    score = model_sub.score(X_test, y_test)
    print(f"  {frac*100:5.0f}% ({n_samples:4d} samples): {score:.4f}")
print("→ Si le score n'augmente pas avec plus de données, le problème est le concept drift, pas la taille.")

---
## 3. Analyse des Biais

In [ ]:
# Analyse complète des biais
bias_results = analyze_bias_comprehensive(best_model, X_test, y_test, df_test)

print("=== ANALYSE DES BIAIS SUR LE SEWOL ===")

for attr_name, attr_data in bias_results.items():
    print(f"\n--- Biais par {attr_name} ---")
    print(f"Groupe de référence : {attr_data['reference_group']}")
    print(f"\n{'Groupe':<15} {'Taux prédit':>12} {'Taux réel':>10} {'DI Ratio':>10} {'Discriminatoire?':>16}")
    print('-' * 68)
    for group, data in attr_data['disparate_impact'].items():
        actual = attr_data['actual_survival'].get(group, 'N/A')
        actual_str = f"{actual:.4f}" if isinstance(actual, (int, float)) else str(actual)
        disc = 'OUI' if data.get('is_discriminatory') else 'Non'
        print(f"{str(group):<15} {data['positive_rate']:>12.4f} {actual_str:>10} "
              f"{data['disparate_impact_ratio']:>10.4f} {disc:>16}")

In [ ]:
# Visualisation des biais
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Analyse des Biais : Prédictions vs Réalité', fontsize=14, fontweight='bold')

y_pred = best_model.predict(X_test)

# Genre
gender_data = pd.DataFrame({
    'Prédiction': df_test.groupby('sex').apply(lambda x: y_pred[x.index - x.index.min()].mean() if len(x) > 0 else 0),
    'Réalité': df_test.groupby('sex')['survived'].mean()
})
# Recalculer proprement
pred_by_sex = {}
real_by_sex = {}
for sex in df_test['sex'].unique():
    mask = df_test['sex'] == sex
    pred_by_sex[sex] = y_pred[mask.values].mean()
    real_by_sex[sex] = y_test[mask].mean()

x_pos = np.arange(len(pred_by_sex))
axes[0].bar(x_pos - 0.2, list(pred_by_sex.values()), 0.4, label='Prédit', color='steelblue')
axes[0].bar(x_pos + 0.2, list(real_by_sex.values()), 0.4, label='Réel', color='coral')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(list(pred_by_sex.keys()))
axes[0].set_title('Biais de Genre')
axes[0].set_ylabel('Taux de survie')
axes[0].legend()
axes[0].axhline(y=0.8, color='red', linestyle='--', alpha=0.3, label='Seuil DI 0.8')

# Classe sociale
pred_by_class = {}
real_by_class = {}
for sc in sorted(df_test['social_class'].unique()):
    mask = df_test['social_class'] == sc
    pred_by_class[sc] = y_pred[mask.values].mean()
    real_by_class[sc] = y_test[mask].mean()

x_pos = np.arange(len(pred_by_class))
axes[1].bar(x_pos - 0.2, list(pred_by_class.values()), 0.4, label='Prédit', color='steelblue')
axes[1].bar(x_pos + 0.2, list(real_by_class.values()), 0.4, label='Réel', color='coral')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'Classe {k}' for k in pred_by_class.keys()])
axes[1].set_title('Biais de Classe Sociale')
axes[1].legend()

# Âge
age_bins = pd.cut(df_test['age'], bins=[0, 18, 30, 50, 100], labels=['0-17', '18-29', '30-49', '50+'])
pred_by_age = {}
real_by_age = {}
for ag in age_bins.unique().dropna():
    mask = age_bins == ag
    if mask.sum() > 0:
        pred_by_age[str(ag)] = y_pred[mask.values].mean()
        real_by_age[str(ag)] = y_test[mask].mean()

x_pos = np.arange(len(pred_by_age))
axes[2].bar(x_pos - 0.2, list(pred_by_age.values()), 0.4, label='Prédit', color='steelblue')
axes[2].bar(x_pos + 0.2, list(real_by_age.values()), 0.4, label='Réel', color='coral')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(list(pred_by_age.keys()))
axes[2].set_title('Biais d\'Âge')
axes[2].legend()

plt.tight_layout()
plt.show()

### Biais dans les données vs biais dans les prédictions

In [ ]:
# Comparer le biais dans les données d'entraînement vs les prédictions
print("=== Biais dans les DONNÉES d'entraînement ===")
print("\nSurvie par genre (Titanic+Lusitania) :")
print(df_train.groupby('sex')['survived'].mean())

print("\nSurvie par classe (Titanic+Lusitania) :")
print(df_train.groupby('social_class')['survived'].mean())

print("\n=== Biais dans les PRÉDICTIONS ===")
print("\nSurvie prédite par genre (Sewol) :")
for sex in df_test['sex'].unique():
    mask = df_test['sex'] == sex
    print(f"  {sex}: prédit={y_pred[mask.values].mean():.4f}, réel={y_test[mask].mean():.4f}")

print("\nSurvie prédite par classe (Sewol) :")
for sc in sorted(df_test['social_class'].unique()):
    mask = df_test['social_class'] == sc
    print(f"  Classe {sc}: prédit={y_pred[mask.values].mean():.4f}, réel={y_test[mask].mean():.4f}")

print("\n→ Le modèle a appris le biais 'femmes d'abord' du Titanic/Lusitania")
print("  et l'applique au Sewol où ce pattern n'existait PAS.")
print("  C'est un cas classique de concept drift lié aux normes sociales.")

---
## 4. Paradoxe Éthique : "Femmes et Enfants d'Abord"

### Le dilemme

Le Titanic (1912) et le Lusitania (1915) ont appliqué la règle « femmes et enfants d'abord » (*Birkenhead drill*). Si notre modèle apprend ce pattern historique avec précision, il sera **biaisé selon les normes modernes d'égalité**.

Trois options s'offrent à nous :

### Option A : Conserver le biais (précision historique)
- **Argument pour** : Le modèle reflète fidèlement la réalité historique. Supprimer le genre réduirait la précision prédictive.
- **Argument contre** : Déployer un tel modèle renforcerait des stéréotypes de genre.

### Option B : Supprimer le biais (équité)
- **Argument pour** : Un modèle équitable ne discrimine pas par genre.
- **Argument contre** : La précision diminue et le modèle ignore un facteur historiquement déterminant.

### Option C : Approche contextuelle (notre choix)

**Notre position** : Conserver le biais historique dans le modèle mais le **documenter explicitement** et **ne pas l'utiliser pour des décisions prospectives**.

**Justification** :
1. **Contexte historique** : Ce modèle analyse des catastrophes passées. Sa valeur est descriptive et explicative, pas prescriptive.
2. **Transparence** : Documenter le biais est plus honnête que de le masquer. Les utilisateurs doivent savoir que le modèle reflète des normes d'une époque révolue.
3. **Séparation prédiction/décision** : Un modèle peut être biaisé dans ses prédictions tout en étant utilisé de manière éthique si les utilisateurs comprennent ses limites.
4. **Impossibilité de neutralité** : Retirer le genre ne supprime pas le biais car d'autres features sont corrélées au genre (classe, titre, taille de groupe). Le modèle reconstruirait indirectement le biais.
5. **Valeur éducative** : Montrer comment le biais historique se propage dans un modèle ML est précisément l'objectif pédagogique de ce projet.

In [ ]:
# Expérience : impact de la suppression du genre
print("=== Expérience : Modèle sans le genre ===")

# Modèle avec genre
model_with_gender = XGBClassifier(n_estimators=200, max_depth=6, random_state=42,
                                  use_label_encoder=False, eval_metric='logloss', verbosity=0)
model_with_gender.fit(X_train, y_train)
score_with = model_with_gender.score(X_test, y_test)

# Modèle sans genre
gender_features = ['sex_binary', 'female_high_class', 'male_low_class']
features_no_gender = [f for f in feature_names if f not in gender_features]
model_no_gender = XGBClassifier(n_estimators=200, max_depth=6, random_state=42,
                                use_label_encoder=False, eval_metric='logloss', verbosity=0)
model_no_gender.fit(X_train[features_no_gender], y_train)
score_without = model_no_gender.score(X_test[features_no_gender], y_test)

print(f"Accuracy avec genre    : {score_with:.4f}")
print(f"Accuracy sans genre    : {score_without:.4f}")
print(f"Différence             : {score_with - score_without:+.4f}")

# Vérifier si le biais persiste sans genre
y_pred_no_gender = model_no_gender.predict(X_test[features_no_gender])
print(f"\nBiais résiduel sans genre (survie prédite par sexe) :")
for sex in df_test['sex'].unique():
    mask = (df_test['sex'] == sex).values
    print(f"  {sex}: {y_pred_no_gender[mask].mean():.4f}")
print("→ Le biais persiste car d'autres features sont corrélées au genre.")

---
## 5. Résumé de l'Évaluation

### Résultats clés

1. **Gap de performance** : Le modèle performe nettement moins bien sur le Sewol que sur Titanic+Lusitania en CV. Ce n'est pas surprenant vu le distribution shift massif.

2. **Sources du gap** :
   - **Covariate shift** : L'âge (dominé par les 17 ans sur le Sewol) et la taille de groupe (classes scolaires) diffèrent radicalement.
   - **Concept drift** : La relation genre→survie est inversée entre 1912 et 2014.
   - **Measurement shift** : Les features contextuelles (température eau, type de naufrage) changent.

3. **Analyse de biais** :
   - Le modèle sur-prédit la survie des femmes sur le Sewol (biais hérité du Titanic).
   - Le biais de classe est moins prononcé car le Sewol n'avait pas de système de classes.
   - Supprimer le genre ne supprime pas complètement le biais (corrélations indirectes).

4. **Position éthique** : Nous conservons le biais historique avec documentation transparente, car le modèle est descriptif et non prescriptif.